# Iteration 7 (CZ port) — image-only pia on CZ stacks

**Goal.** Apply the iter07 HCR detector — originally built and tuned on HCR `load_hcr_combined` volumes — to CZ (confocal) stacks of the same 6 subjects.  One detector that works on both modalities is the minimum bar for promoting iter07 into a public API.

**Status after four sub-iterations (this notebook).**

1. **v0 — identical protocol**: range-relative threshold with `TRANS_FRAC=0.5`.  Passed the first visual check, then **failed** on closer inspection.  Red consistently ~10 µm below the visible tissue boundary, and catastrophically wrong on 767022 (red at 102 µm inside dense tissue, not at the ~50 µm onset).
2. **v1 — noise-anchored threshold**: `thr = col_p10 + 3·MAD`.  *Worse* than v0 — 767022 degenerated to 208 µm with 166/400 valid columns.
3. **v2 — explicit bg-subtraction**: estimate camera offset, clip, log.  Offset mode came out at ~1–5 DN (padding zeros dominate the histogram mode), numerically no change vs v0.
4. **v3 — floor-hugging TRANS_FRAC=0.02**: **fixed 767022** (102 → 46 µm) and moved every other subject ~10 µm shallower.  Red within ~10 µm of visible onset on 5/6, 767022 at visible onset.

**Root cause (diagnosed in v1/v2).**  HCR and CZ have different log-intensity structure:

| | HCR combined | CZ raw |
|---|---|---|
| OOT voxel value | ≡ 0 (bg-subtracted) | 0 or small positive (mostly) |
| `log(OOT+ε)` | hard floor at −6.9 | no hard floor |
| OOT → tissue | **cliff** (several log units in a few µm) | **ramp** (gradual rise over tens of µm) |
| col_p10 in ramp subjects | OOT floor (−6.9) | inside the ramp |

`thr = col_p10 + 0.5·(col_p90 − col_p10)` on HCR hits the cliff's foot because col_p10 IS the floor.  On CZ, col_p10 already sits in the ramp, so the 50 % midpoint lands mid-tissue.  Lowering `TRANS_FRAC` to 0.02 makes thr ≈ col_p10, firing at the first sustained rise — which is what HCR achieves by accident.

## v0 — identical protocol (`TRANS_FRAC = 0.5`) — failed

Drop-in of the HCR pipeline (patch-MAX detector + IRLS-Huber poly-deg2) with only `THR_FLOOR` retuned as a sanity floor (−6.3 → 4.5).  Script: `iter07_cz_proto.py`.  First bench looked plausible — 400/400 valid everywhere — but visual inspection of the raw-vs-fit panels (next cell) revealed the red surface sits below visible tissue onset on all 6 subjects, with 767022 dramatically inside the tissue layer.

In [ ]:
from pathlib import Path
from IPython.display import Image, display, Markdown
import pandas as pd

SESSION = Path('/root/capsule/code/sessions/03c_onset_features')
FIG = SESSION / 'figures'
DATA = SESSION / 'data'
SUBJECTS = ['755252', '767018', '767022', '782149', '788406', '790322']

df0 = pd.read_csv(DATA / 'iter07_cz_summary.csv')
df0['subject'] = df0.subject.astype(str)
print('v0 — TRANS_FRAC=0.5 (identical to HCR protocol) — FAILED')
print(df0[['subject', 'log_p10', 'log_p90', 'median_thr',
           'n_valid_trans', 'median_trans_z_um']].to_string(
    index=False, float_format=lambda x: f'{x:7.2f}'))

In [ ]:
# v0 side-by-side — failing result, especially catastrophic on 767022
for sid in SUBJECTS:
    display(Markdown(f'### v0 — {sid}'))
    display(Image(str(FIG / f'iter07_cz_sidebyside_{sid}.png')))

## v1 — noise-anchored threshold — worse than v0

**Hypothesis.** HCR's col_p10 happens to be pure-OOT noise because combined volumes are bg-subtracted; CZ's col_p10 sits inside the tissue ramp.  Replace the range-relative rule with a *noise-anchored* one: `thr = max(THR_FLOOR, col_p10 + 3 · MAD_lower)` where `MAD_lower` is 1.4826 × MAD over smoothed values ≤ the 20th percentile (tight bottom tail = pure-OOT estimate).

**Result.** Catastrophically worse.  MAD estimated from "smooth ≤ p20" still spans a large portion of the OOT → tissue ramp because OOT is only 5–15 % of a typical CZ column, so p20 falls inside the ramp.  Inflated MAD → inflated σ → threshold pushed **further into tissue** than v0.  767022 degenerated to 208 µm with 166/400 valid columns (many columns never cross the now-too-high threshold).  Script: `iter07_cz_noise.py`.

**Lesson.** Any column-level noise estimator needs to isolate pure-OOT voxels, which is hard on CZ because OOT occupies a minority of the z-range.  Abandoned this branch.

In [ ]:
df1 = pd.read_csv(DATA / 'iter07_cz_noise_summary.csv')
df1['subject'] = df1.subject.astype(str)
print('v1 — noise-anchored (thr = col_p10 + 3·MAD_lower) — WORSE')
print(df1[['subject', 'median_thr', 'n_valid_trans',
           'median_trans_z_um']].to_string(
    index=False, float_format=lambda x: f'{x:7.2f}'))

In [ ]:
# v1 side-by-side — 755252 clearly too deep, 767022 far below tissue onset
for sid in ['755252', '767022']:
    display(Markdown(f'### v1 — {sid}'))
    display(Image(str(FIG / f'iter07_cz_noise_{sid}.png')))

## v2 — explicit bg-subtraction — numerically no-op

**Hypothesis.** CZ has a camera offset (~100 DN) that we hadn't removed.  If we estimate offset (histogram mode) and subtract it, then clip at 0 before log, CZ voxels below the offset land at `log(ε) = −6.9` — restoring the HCR-style cliff floor.  Then the range-relative rule with `TRANS_FRAC = 0.5` should work unchanged.

**Result.** Offset came out at 1.0 – 5.4 DN across subjects (not 100 as expected).  Median and log-percentile tables below are essentially identical to v0.  Why: the CZ TIFF is already bg-subtracted somewhere upstream — the histogram mode is dominated by **true-zero padding voxels** from coregistration, not by a camera-offset peak.  The OOT voxels in our volumes have real continuous values in the low-DN range; they're not clustered at a pedestal.

**Lesson.** CZ doesn't have the HCR-style binary OOT/tissue separation at all.  It's a density ramp — tissue gets denser with depth, no cliff.  Trying to *create* a cliff via a global offset doesn't work because no such offset exists.  Script: `iter07_cz_bgsub.py`.

In [ ]:
df2 = pd.read_csv(DATA / 'iter07_cz_bgsub_summary.csv')
df2['subject'] = df2.subject.astype(str)
print('v2 — hist-mode bg-subtract + TRANS_FRAC=0.5 — NO CHANGE')
print(df2[['subject', 'offset', 'log_p10', 'log_p90',
           'median_thr', 'median_trans_z_um']].to_string(
    index=False, float_format=lambda x: f'{x:7.2f}'))

## v3 — floor-hugging `TRANS_FRAC = 0.02` — works

**Hypothesis (final).**  The problem isn't the threshold *rule* (range-relative is correct); it's the threshold *fraction*.  On HCR, the p10→p90 range is dominated by the several-log-unit cliff between OOT (−6.9) and tissue.  Any `TRANS_FRAC` between ~0.01 and 0.5 produces a threshold that lives *in the cliff* and fires at the cliff's foot — they all look the same.  On CZ, the p10→p90 range is just the tissue ramp; `TRANS_FRAC = 0.5` picks a point mid-tissue.

Fix: lower `TRANS_FRAC` to 0.02, matching `image_ceiling`'s `relative_margin=0.02`.  `thr ≈ col_p10` — fires at first sustained rise above the column's own floor.

**Result.** 767022 fixed (102 → 46 µm).  Every subject shifted 8–56 µm shallower.  All 6 at 400/400 valid.  Red within ~10 µm of visible tissue onset on 5/6 and right at onset on 767022.  Script: `iter07_cz_lowfrac.py`.

In [ ]:
df3 = pd.read_csv(DATA / 'iter07_cz_lowfrac_summary.csv')
df3['subject'] = df3.subject.astype(str)
merged = df0[['subject', 'median_trans_z_um']].rename(
    columns={'median_trans_z_um': 'v0_trans_z'}).merge(
    df3[['subject', 'median_trans_z_um', 'median_thr']].rename(
        columns={'median_trans_z_um': 'v3_trans_z',
                 'median_thr': 'v3_thr'}),
    on='subject')
merged['delta_z_um'] = merged.v3_trans_z - merged.v0_trans_z
print('v0 (TRANS_FRAC=0.5) vs v3 (TRANS_FRAC=0.02)')
print(merged.to_string(index=False, float_format=lambda x: f'{x:7.2f}'))

In [ ]:
# v3 side-by-side — the recommended setting
for sid in SUBJECTS:
    display(Markdown(f'### v3 — {sid}'))
    display(Image(str(FIG / f'iter07_cz_lowfrac_{sid}.png')))

## Bottom line + open questions

**What worked.** Same detector + same range-relative rule as HCR, with `TRANS_FRAC = 0.02` on CZ (0.5 on HCR).  On HCR, 0.02 vs 0.5 don't differ materially because the HCR cliff dominates the range; on CZ, the lower value is essential.  A single global `TRANS_FRAC = 0.02` would probably work on both, worth regression-checking HCR under 0.02 before any API promotion.

**What didn't.** Noise-anchored MAD (v1) — requires cleaner OOT isolation than CZ affords.  Explicit bg-subtraction (v2) — CZ has no camera-offset pedestal to remove.

**Remaining gap.** v3 red still sits ~5–15 µm below the visible tissue boundary on most subjects.  Two plausible contributors:

1. **Anchor percentile too high.** col_p10 already sits inside the ramp on CZ because OOT is only 5–15 % of the column.  Using col_p3 or col_p5 as the anchor would drop thr closer to the true floor.
2. **Sustain latency.** `SUSTAIN_Z_UM = 15` requires 15 µm of continuous above-threshold signal, adding ~7–10 µm to the detected onset.  Shortening would help but sacrifice noise rejection.

**Next decision** (user call): accept v3 as-is (good enough for curvature-capable image-only pia on CZ, with the AF-edge case 767022 fixed), or chase the remaining ~10 µm via lower-percentile anchor or shorter sustain.  Also open: whether to switch HCR to `TRANS_FRAC = 0.02` so the protocol is truly modality-agnostic.